## Delete emptied author profiles (oxjob #1354)

Runs after `Create_Authors`. A profile is **empty** when `openalex_authors.works_count = 0` **and** it holds no
`openalex.works.work_authors` row. Empty profiles are tracked in `zero_works_tracking`; once one has been empty for
`guard_days` consecutive runs and is not on the hold list (author claims, work-author claim curations), it is ledgered in
`openalex.authors.deleted_authors` (row kept verbatim for undo) and deleted from `authors` and `openalex_authors`.
The ES doc is removed by `notebooks/elastic/delete_authors` on the next authors sync. Hard delete, 404, no redirects,
no `deleted_ids` export (decision 2026-09-24).

Knobs are the three variables below (re-declared in every cell that uses them, since each cell is its own statement
batch on the warehouse). `dry_run = true` computes and reports but deletes nothing; flip it here, in code.
Backlog drains through this job at `max_deletes_per_run` per day, oldest `first_zero_date` first.

In [ ]:
-- Upsert today's empty set: zero works AND no seat anywhere.
MERGE INTO openalex.authors.zero_works_tracking t
USING (
  SELECT oa.id AS author_id, current_date() AS today
  FROM openalex.authors.openalex_authors oa
  LEFT ANTI JOIN openalex.works.work_authors wa ON wa.author_id = oa.id
  WHERE oa.works_count = 0
) s ON t.author_id = s.author_id
WHEN MATCHED     THEN UPDATE SET last_seen_zero_date = s.today
WHEN NOT MATCHED THEN INSERT (author_id, first_zero_date, last_seen_zero_date) VALUES (s.author_id, s.today, s.today);

In [ ]:
-- Evict rebounds: not seen empty today (regained a work or a seat) means the clock restarts next time.
DELETE FROM openalex.authors.zero_works_tracking
WHERE last_seen_zero_date < current_date();

In [ ]:
DECLARE OR REPLACE VARIABLE guard_days INT DEFAULT 7;
DECLARE OR REPLACE VARIABLE max_deletes_per_run INT DEFAULT 2000000;
DECLARE OR REPLACE VARIABLE dry_run BOOLEAN DEFAULT true;

-- Candidates: past the guard, not held, capped per run, oldest first.
-- Hold list: any author claim (approved or pending) or any work-author claim curation pointing at the profile.
-- (Two statements: session variables cannot be referenced inside CREATE TABLE AS SELECT, but INSERT ... SELECT is fine.)
CREATE OR REPLACE TABLE openalex.authors.deleted_authors_candidates (author_id BIGINT, first_zero_date DATE);
INSERT INTO openalex.authors.deleted_authors_candidates
SELECT t.author_id, t.first_zero_date
FROM openalex.authors.zero_works_tracking t
LEFT ANTI JOIN (
  SELECT TRY_CAST(REGEXP_EXTRACT(LOWER(author_id), 'a(\\d+)', 1) AS BIGINT) AS author_id
  FROM openalex_users.public.author_claims
  WHERE decision IN ('approved', 'pending')
  UNION
  SELECT author_id FROM openalex.works.work_author_claim_curations
) hold ON hold.author_id = t.author_id
WHERE DATEDIFF(current_date(), t.first_zero_date) >= guard_days
QUALIFY ROW_NUMBER() OVER (ORDER BY t.first_zero_date, t.author_id) <= max_deletes_per_run;

In [ ]:
DECLARE OR REPLACE VARIABLE guard_days INT DEFAULT 7;
DECLARE OR REPLACE VARIABLE max_deletes_per_run INT DEFAULT 2000000;
DECLARE OR REPLACE VARIABLE dry_run BOOLEAN DEFAULT true;

-- Report. Read this before trusting a run.
SELECT
  (SELECT COUNT(*) FROM openalex.authors.zero_works_tracking) AS tracked_empty,
  (SELECT COUNT(*) FROM openalex.authors.zero_works_tracking WHERE DATEDIFF(current_date(), first_zero_date) >= guard_days) AS past_guard,
  (SELECT COUNT(*) FROM openalex.authors.deleted_authors_candidates) AS will_delete_this_run,
  (SELECT MIN(first_zero_date) FROM openalex.authors.deleted_authors_candidates) AS oldest_candidate,
  (SELECT COUNT(*) FROM openalex.authors.deleted_authors WHERE es_deleted_at IS NULL) AS awaiting_es_delete,
  guard_days, max_deletes_per_run, dry_run;

In [ ]:
-- Sanity: never more than 3% of the table in one run (throttle bug or a bad rebuild upstream).
SELECT CASE
  WHEN (SELECT COUNT(*) FROM openalex.authors.deleted_authors_candidates)
     > 0.03 * (SELECT COUNT(*) FROM openalex.authors.authors)
  THEN RAISE_ERROR('DeleteEmptiedAuthors: candidates exceed 3% of openalex.authors.authors; lower max_deletes_per_run or investigate')
END AS sanity;

In [ ]:
DECLARE OR REPLACE VARIABLE guard_days INT DEFAULT 7;
DECLARE OR REPLACE VARIABLE max_deletes_per_run INT DEFAULT 2000000;
DECLARE OR REPLACE VARIABLE dry_run BOOLEAN DEFAULT true;

-- Ledger first: the profile row verbatim, so undo is an INSERT back into authors. No-op when dry_run.
INSERT INTO openalex.authors.deleted_authors
SELECT a.id, a.display_name, a.full_name, a.orcid, a.created_date, a.updated_date,
       c.first_zero_date, 'guard', current_timestamp(), NULL
FROM openalex.authors.deleted_authors_candidates c
JOIN openalex.authors.authors a ON a.id = c.author_id
LEFT ANTI JOIN openalex.authors.deleted_authors d ON d.author_id = c.author_id
WHERE NOT dry_run;

In [ ]:
DECLARE OR REPLACE VARIABLE guard_days INT DEFAULT 7;
DECLARE OR REPLACE VARIABLE max_deletes_per_run INT DEFAULT 2000000;
DECLARE OR REPLACE VARIABLE dry_run BOOLEAN DEFAULT true;

-- Delete from the source of truth.
MERGE INTO openalex.authors.authors a
USING openalex.authors.deleted_authors_candidates c ON a.id = c.author_id
WHEN MATCHED AND NOT dry_run THEN DELETE;

In [ ]:
DECLARE OR REPLACE VARIABLE guard_days INT DEFAULT 7;
DECLARE OR REPLACE VARIABLE max_deletes_per_run INT DEFAULT 2000000;
DECLARE OR REPLACE VARIABLE dry_run BOOLEAN DEFAULT true;

-- And from the rollup, so the snapshot export and the next ES index pass never see the row again.
MERGE INTO openalex.authors.openalex_authors oa
USING openalex.authors.deleted_authors_candidates c ON oa.id = c.author_id
WHEN MATCHED AND NOT dry_run THEN DELETE;

In [ ]:
DECLARE OR REPLACE VARIABLE guard_days INT DEFAULT 7;
DECLARE OR REPLACE VARIABLE max_deletes_per_run INT DEFAULT 2000000;
DECLARE OR REPLACE VARIABLE dry_run BOOLEAN DEFAULT true;

-- Tracking rows for deleted ids are done.
MERGE INTO openalex.authors.zero_works_tracking t
USING openalex.authors.deleted_authors_candidates c ON t.author_id = c.author_id
WHEN MATCHED AND NOT dry_run THEN DELETE;

In [ ]:
-- Verify: no ledgered id may still resolve.
SELECT CASE
  WHEN EXISTS (SELECT 1 FROM openalex.authors.deleted_authors d JOIN openalex.authors.authors a ON a.id = d.author_id)
  THEN RAISE_ERROR('DeleteEmptiedAuthors: a ledgered id still exists in openalex.authors.authors')
END AS verify;